# Arbitrary precision with SpinWeightedSpheroidalHarmonics.jl

Every eigenvalue, harmonic and continued mode is computed in the floating-point type of its input:
`Float64` by default, `BigFloat` at any number of bits. This notebook shows how to ask for more digits,
what the results look like, how accurate they are, and what they cost.

In [1]:
using Pkg; Pkg.activate(joinpath(@__DIR__, ".."))
using SpinWeightedSpheroidalHarmonics, Printf
setprecision(BigFloat, 256);   # the working precision used below unless stated

  Activating project at `~/Documents/GitHub/SpinWeightedSpheroidalHarmonics.jl`


## 1. Three ways to ask for more digits

**`Float64`** is the default. **`BigFloat` input** computes at the precision of the input. **`precision = p`**
converts the input to `p`-bit `BigFloat`; a `Rational` is then rounded once, so `7//20` is exactly
``7/20`` to the last bit.

In [2]:
s, l, m = -2, 2, 2
λ64  = spin_weighted_spheroidal_eigenvalue(s, l, m, 0.35)                   # Float64
λbig = spin_weighted_spheroidal_eigenvalue(s, l, m, parse(BigFloat, "0.35")) # BigFloat input
λkw  = spin_weighted_spheroidal_eigenvalue(s, l, m, 7//20; precision=256)    # keyword, exact 7/20
@show λ64 typeof(λ64)
@show λkw typeof(λkw) precision(λkw)
λbig == λkw

λ64 = 1.6966094016353415
typeof(λ64) = Float64
λkw = 1.696609401635341560961986869798062865250760453128198253695628864777620341839445
typeof(λkw) = BigFloat
precision(λkw) = 256


true

The same works for the harmonic. The returned object evaluates at its own precision, and the angles
can be given as `Float64` (exact binary values), `BigFloat` or rationals converted beforehand.

In [3]:
S = spin_weighted_spheroidal_harmonic(s, l, m, 7//20; precision=256)
θ, φ = big(11)/10, big(3)/10
@show S(θ, φ)
@show S(θ, φ; theta_derivative=1)
@show S(θ, φ; theta_derivative=2, phi_derivative=1);

S(θ, φ) = 0.2597765692043830005986381707511224952143972595844395972191350831069059365250316 + 0.1777227129374413434911956429326069627356280538994608854506996018351625413563537im
S(θ, φ; theta_derivative = 1) = -0.3749860844330337167122847434370035515194739246853157539198255970803931125628542 - 0.2565417829765640407671957053459873369892198755380579870307281517880951908819507im
S(θ, φ; theta_derivative = 2, phi_derivative = 1) = -0.2146664530434268612565596603009031446208986099769464910236346828074710381785873 + 0.3137770843872087675313838700223817066113626917569352663773151950151496023651742im


## 2. Accuracy follows the precision

The eigenvalue for ``(s, \ell, m) = (-2, 2, 2)``, ``c = 1/2`` computed at increasing precision, compared with
512 bits, and with the Mathematica BHPToolkit value (`SpinWeightedSpheroidalEigenvalue[-2,2,2,1/2]`, 60 digits).

In [4]:
bhpt = parse(BigFloat, "0.725702761257765110195306308936146146138102156529773250230740"; precision=256)
ref = spin_weighted_spheroidal_eigenvalue(-2, 2, 2, 1//2; precision=512)
@printf "%-8s %-12s %-12s\n" "bits" "|λ - λ512|" "|λ - BHPT|"
for bits in (53, 128, 256)
    λ = bits == 53 ? spin_weighted_spheroidal_eigenvalue(-2, 2, 2, 0.5) :
                     spin_weighted_spheroidal_eigenvalue(-2, 2, 2, 1//2; precision=bits)
    @printf "%-8d %-12.2e %-12.2e\n" bits Float64(abs(big(λ) - ref)) Float64(abs(big(λ) - bhpt))
end
# the BHPToolkit value has 60 digits, so it agrees to ~1e-60 at most

bits     |λ - λ512|   |λ - BHPT|  
53       3.39e-16     3.39e-16    
128      5.42e-37     5.42e-37    
256      9.17e-75     2.83e-61    


The same ladder for harmonic values and derivatives, for every method:

In [5]:
function ladder(method; c=1//2, θ=big(11)/10)
    v(bits) = (h = spin_weighted_spheroidal_harmonic(-2, 2, 2, c; method, precision=bits);
               [setprecision(() -> h(θ, big(3)/10; theta_derivative=d), BigFloat, 600) for d in 0:2])
    v512 = v(512)
    err(v) = maximum(Float64(abs(a - b) / max(1, abs(b))) for (a, b) in zip(v, v512))
    h64 = spin_weighted_spheroidal_harmonic(-2, 2, 2, Float64(c); method)
    e64 = maximum(Float64(abs(h64(Float64(θ), 0.3; theta_derivative=d) - v512[d+1]) / max(1, abs(v512[d+1]))) for d in 0:2)
    @printf "%-10s Float64 %.1e   128 bits %.1e   256 bits %.1e\n" method e64 err(v(128)) err(v(256))
end
for method in ("auto", "jacobi", "chebyshev", "leaver"); ladder(method); end

auto       Float64 4.9e-17   128 bits 2.4e-38   256 bits 7.8e-78
jacobi     Float64 4.9e-17   128 bits 2.4e-38   256 bits 7.8e-78
chebyshev  Float64 2.1e-14   128 bits 1.1e-35   256 bits 4.1e-73
leaver     Float64 3.2e-16   128 bits 9.6e-40   256 bits 1.7e-78


## 3. A harmonic remembers its precision

Built at 256 bits, it evaluates at 256 bits even where the caller has set a lower precision, and it
can be evaluated from several threads at once.

In [6]:
v = setprecision(BigFloat, 64) do
    S(1.1, 0.3)          # Float64 angles, caller at 64 bits
end
@show typeof(v) precision(real(v));

typeof(v) = Complex{BigFloat}
precision(real(v)) = 256


## 4. Complex spheroidicity and continued modes

Complex `c` works the same way. The mode is followed from ``c = 0`` along the path (in `Float64`), then
every state is refined in the working precision. The states of a path keep the phase of the continuation;
a harmonic fixes its overall phase by making the overlap with ``{}_sY_{\ell m}`` real and positive, so every
method (spectral, Chebyshev, Leaver) returns the same function. It is normalized with ``|S|^2``; the
Mathematica BHPToolkit package normalizes ``S^2`` instead (see the documentation home page).

In [7]:
c = 1//2 - 1im//2
λc = spin_weighted_spheroidal_eigenvalue(-2, 2, 2, c; precision=256)
pair = continue_angular_mode(-2, 2, 2, c; precision=256)
@show λc
@show pair.precision_bits pair.residual
Sc = spin_weighted_spheroidal_harmonic(pair)
@show Sc(1.1, 0.3);
path = track_angular_mode(-2, 2, 2, [0, 1//4 - 1im//4, 1//2 - 1im//2]; precision=256)
[(Float64(abs(st.c)), Float64(st.residual)) for st in path.states]

λc = 0.6803519763559509378298914572843760339850437435472451568548432788949557259606982 + 3.215940703201343886779245549634596951270707230026174605554031088416770460065601im
pair.precision_bits = 256
pair.residual = 8.191985393069370639888414510789521698375573324481460135346874577781324230855372e-81
Sc(1.1, 0.3) = 0.2368038281812443713684710520596441289664098648670354874666018740446883968883909 + 0.1961171865523142787335287224030413451490790200656153754339233630100323391380895im


3-element Vector{Tuple{Float64, Float64}}:
 (0.0, 0.0)
 (0.3535533905932738, 7.940908986747958e-81)
 (0.7071067811865476, 8.19198539306937e-81)

In [8]:
# all methods give the same function for complex c, values and θ-derivatives alike
for method in ("leaver", "chebyshev")
    h = spin_weighted_spheroidal_harmonic(-2, 2, 2, c; method, precision=256)
    @printf "%-10s max |h - auto| over θ, d=0..2: %.1e\n" method maximum(Float64(abs(h(θ, 0.3; theta_derivative=d) - Sc(θ, 0.3; theta_derivative=d))) for θ in (0.0, 0.5, 1.1, 2.0, π), d in 0:2)
end

leaver     max |h - auto| over θ, d=0..2: 3.4e-76
chebyshev  max |h - auto| over θ, d=0..2: 2.2e-69


### Complex frequency: accuracy follows the precision

``c = a\omega`` with complex ``\omega`` (quasinormal-mode-like values and larger ``|c|``), default method. Errors against
512 bits: ``\lambda`` relative, and the harmonic with its first two ``\theta``-derivatives over ``\theta`` from pole to pole,
relative to the size of each derivative.

In [9]:
θs = (0.0, 1e-8, 0.3, 1.1, π/2, 2.9, π)
cases = [("(-2,2,2) a=0.7, ω=0.5326-0.0808i", -2, 2, 2, 7//10 * (5326//10000 - 808im//10000)),
         ("(-2,2,2) a=0.9, ω=0.64-0.30i",     -2, 2, 2, 9//10 * (64//100 - 30im//100)),
         ("(-2,3,1) c=1-2i",                  -2, 3, 1, 1 - 2im),
         ("(-2,20,2) c=-1.5-0.7i",            -2, 20, 2, -3//2 - 7im//10)]
@printf "%-34s %-10s %-10s %-10s %-10s\n" "case" "" "Float64" "128 bits" "256 bits"
for (name, s, l, m, c) in cases
    ref = spin_weighted_spheroidal_harmonic(s, l, m, c; precision=512)
    R = Dict((θ, d) => setprecision(() -> ref(BigFloat(θ), BigFloat(0.3); theta_derivative=d), BigFloat, 512) for θ in θs, d in 0:2)
    sc = [maximum(abs(R[(θ, d)]) for θ in θs) for d in 0:2]
    eλ, eh = String[], String[]
    for bits in (53, 128, 256)
        h = bits == 53 ? spin_weighted_spheroidal_harmonic(s, l, m, ComplexF64(c)) :
                         spin_weighted_spheroidal_harmonic(s, l, m, c; precision=bits)
        T(x) = bits == 53 ? x : setprecision(() -> BigFloat(x), BigFloat, bits)
        vals = Dict((θ, d) => h(T(θ), T(0.3); theta_derivative=d) for θ in θs, d in 0:2)
        setprecision(BigFloat, 512) do
            push!(eλ, @sprintf("%.0e", Float64(abs(Complex{BigFloat}(h.lambda) - ref.lambda) / abs(ref.lambda))))
            push!(eh, @sprintf("%.0e", maximum(Float64(abs(Complex{BigFloat}(vals[(θ, d)]) - R[(θ, d)]) / sc[d+1]) for θ in θs, d in 0:2)))
        end
    end
    @printf "%-34s %-10s %-10s %-10s %-10s\n" name "λ" eλ...
    @printf "%-34s %-10s %-10s %-10s %-10s\n" "" "S,∂θ,∂²θ" eh...
end

case                                          Float64    128 bits   256 bits  
(-2,2,2) a=0.7, ω=0.5326-0.0808i   λ          3e-16      3e-39      2e-77     
                                   S,∂θ,∂²θ   2e-15      2e-38      4e-76     
(-2,2,2) a=0.9, ω=0.64-0.30i       λ          4e-16      6e-39      1e-77     
                                   S,∂θ,∂²θ   1e-14      3e-37      6e-77     
(-2,3,1) c=1-2i                    λ          2e-14      1e-38      5e-77     
                                   S,∂θ,∂²θ   2e-14      1e-37      2e-76     
(-2,20,2) c=-1.5-0.7i              λ          2e-16      1e-39      3e-77     
                                   S,∂θ,∂²θ   9e-14      2e-37      1e-75     


## 5. Modes that Float64 cannot resolve

At large negative ``mc`` the ``\ell = 2`` mode and its ``\ell = 3`` neighbour are closer than Float64 can
tell apart (relative gap ~``10^{-13}`` at ``c = -20``), so Float64 refuses to return the harmonic. A wider
precision selects the mode and resolves it.

In [10]:
try
    spin_weighted_spheroidal_harmonic(-2, 2, 2, -20.0)
catch err
    println(first(sprint(showerror, err), 220), "…")
end

The SWSH eigenvector for (s, l, m) = (-2, 2, 2), c = -20.0 cannot be resolved in Float64: the mode is nearly degenerate with a neighbouring mode (relative gap 2.1282756917250925e-14, below 2.220446049250313e-10), so the …


In [11]:
h = spin_weighted_spheroidal_harmonic(-2, 2, 2, -20; precision=256)
λ3 = spin_weighted_spheroidal_eigenvalue(-2, 3, 2, -20; precision=256)
@show Float64(abs(h.lambda - λ3) / abs(λ3))      # relative gap to the neighbour
@show h(1.1, 0.3);

Float64(abs(h.lambda - λ3) / abs(λ3)) = 2.9606085483189067e-13
h(1.1, 0.3) = 6.789327171911664710585800796926806927430787892959112076109874531562862159541028e-05 + 4.64482862217917426447708172611192452977041240364848917355519476443992473028687e-05im


## 6. Float64 at large ``\ell``

The default method evaluates the spherical harmonics with the Jacobi recurrence for every ``\ell``. The direct
sum (`method="direct"`) loses digits to cancellation as ``\ell`` grows; the Jacobi recurrence does not, and it is
also much faster. Errors of the Float64 harmonic and its first two ``\theta``-derivatives, pole to pole, against
256 bits:

In [12]:
θs = (0.0, 1e-8, 1e-3, 0.3, 1.1, π/2, 2.9, π)
@printf "%-26s %-24s %-24s\n" "(s,l,m,c)" "direct  S/∂θ/∂²θ" "default S/∂θ/∂²θ"
for (s, l, m, c) in ((-2, 2, 2, 1//2), (1, 15, -4, 2), (-2, 20, 2, -3//2 - 7im//10), (-1, 29, 0, 3//10))
    ref = spin_weighted_spheroidal_harmonic(s, l, m, c; precision=256)
    cf = c isa Complex ? ComplexF64(c) : Float64(c)
    errs(h) = join((@sprintf("%.0e", maximum(Float64(abs(h(θ, 0.3; theta_derivative=d) - ref(big(θ), big(0.3); theta_derivative=d))) for θ in θs) /
        Float64(maximum(abs(ref(big(t), big(0.3); theta_derivative=d)) for t in 0.05:0.15:3.1))) for d in 0:2), "/")
    @printf "%-26s %-24s %-24s\n" string((s, l, m, cf)) errs(spin_weighted_spheroidal_harmonic(s, l, m, cf; method="direct")) errs(spin_weighted_spheroidal_harmonic(s, l, m, cf))
end

(s,l,m,c)                  direct  S/∂θ/∂²θ         default S/∂θ/∂²θ        
(-2, 2, 2, 0.5)            5e-16/3e-16/6e-16        5e-16/2e-16/6e-16       
(1, 15, -4, 2.0)           1e-12/1e-12/3e-13        3e-15/1e-15/9e-16       
(-2, 20, 2, -1.5 - 0.7im)  2e-10/7e-11/4e-12        1e-13/5e-14/1e-13       
(-1, 29, 0, 0.3)           1e-07/5e-08/5e-09        7e-15/2e-14/6e-15       


## 7. Derivatives at the poles, every method

``\theta``-derivatives are exact at ``\theta = 0`` and ``\pi`` for every method (Leaver's are taken from the
Taylor coefficients of its series, with no fit). The Chebyshev solver loses a few digits in its derivatives at the endpoints; the default method does not.
Float64, against 256 bits, for a complex frequency:

In [13]:
s, l, m, c = -2, 3, 1, 1//2 - 1im//2
ref = spin_weighted_spheroidal_harmonic(s, l, m, c; precision=256)
@printf "%-10s %-10s %-10s %-10s\n" "method" "∂θ at 0" "∂²θ at 0" "∂²θ at π"
for method in ("auto", "chebyshev", "leaver")
    h = spin_weighted_spheroidal_harmonic(s, l, m, ComplexF64(c); method)
    e(θ, d) = Float64(abs(h(θ, 0.3; theta_derivative=d) - ref(big(θ), big(0.3); theta_derivative=d)) /
        maximum(abs(ref(big(t), big(0.3); theta_derivative=d)) for t in 0.05:0.15:3.1))
    @printf "%-10s %-10.0e %-10.0e %-10.0e\n" method e(0.0, 1) e(0.0, 2) e(Float64(π), 2)
end

method     ∂θ at 0    ∂²θ at 0   ∂²θ at π  
auto       2e-16      0e+00      6e-31     
chebyshev  8e-12      3e-10      9e-12     
leaver     1e-15      0e+00      6e-32     


## 8. Comparing with the BHPToolkit Mathematica package

For real ``c`` both packages use ``\int_0^\pi S^2 \sin\theta\, d\theta = 1/(2\pi)`` and agree directly. For complex ``c``
this package normalizes ``|S|^2`` and makes the overlap with ``{}_sY_{\ell m}`` real and positive; BHPToolkit
normalizes ``S^2`` without the complex conjugate. Its harmonic is this one divided by
``\sqrt{\sum_{\ell'} b_{\ell'}^2}``, with ``b_{\ell'}`` the spherical-harmonic coefficients (`S.coeffs`), up to the sign of the square root:

In [14]:
S = spin_weighted_spheroidal_harmonic(-2, 2, 2, 1//2 - 1im//2; precision=256)
to_bhpt = 1 / sqrt(sum(b^2 for b in S.coeffs))
@show to_bhpt
@show S(big(11)/10, big(0)) * to_bhpt;     # the value BHPToolkit's SpinWeightedSpheroidalHarmonicS gives

to_bhpt = 1.009013137781182132534786705722996332136956018303468506416294831346187532028586 + 0.009140369470351675810145988830713118082896652383779490464518602457394879448388241im
S(big(11) / 10, big(0)) * to_bhpt = 0.3086810286649702468497950827734259987178406864922884760645714802235398746523478 + 0.03120533305089987173111611365072250875351995504540719850780837240879118602831209im


## 9. Spherical harmonics

`spin_weighted_spherical_harmonic` takes the same keyword; without it, it evaluates in the type of the angles.

In [15]:
Y = spin_weighted_spherical_harmonic(-2, 31, 1; precision=256)
@show Y(1e-8, 0.0)                     # near the pole, now accurate in Float64 too
@show spin_weighted_spherical_harmonic(-2, 31, 1)(1e-8, 0.0);

Y(1.0e-8, 0.0) = 3.522515755933759303246731218162459324652135262865073122562727242467473277497666e-07 + 0.0im
(spin_weighted_spherical_harmonic(-2, 31, 1))(1.0e-8, 0.0) = 3.522515755933803e-7 + 0.0im


## 10. Threads

Building and evaluating from several threads at once gives the same bits as serial evaluation, at any
precision (start Julia with `--threads=N` to use more than one).

In [16]:
cs = [k // 10 for k in 1:6]
serial = [spin_weighted_spheroidal_eigenvalue(-2, 2, 2, c; precision=256) for c in cs]
threaded = fetch.([Threads.@spawn spin_weighted_spheroidal_eigenvalue(-2, 2, 2, c; precision=256) for c in cs])
@show Threads.nthreads() serial == threaded;

Threads.nthreads() = 4
serial == threaded = true


## 11. Cost

`BigFloat` is roughly a thousand times slower than `Float64`, and the cost grows with the number of
bits. Timings after compilation on this machine (construction of the harmonic, one evaluation):

In [17]:
function cost(bits; c=1//2, method="auto")
    cc = bits == 53 ? Float64(c) : c; kw = bits == 53 ? (;) : (; precision=bits)
    spin_weighted_spheroidal_harmonic(-2, 2, 2, cc; method, kw...)
    tb = @elapsed h = spin_weighted_spheroidal_harmonic(-2, 2, 2, cc; method, kw...)
    h(1.1, 0.3)
    te = @elapsed for _ in 1:10; h(1.1, 0.3); end
    tb, te / 10
end
@printf "%-10s %-6s %-14s %-14s\n" "method" "bits" "build (s)" "evaluate (s)"
for method in ("auto", "leaver", "chebyshev"), bits in (53, 128, 256, 512)
    tb, te = cost(bits; method)
    @printf "%-10s %-6d %-14.2e %-14.2e\n" method bits tb te
end

method     bits   build (s)      evaluate (s)  
auto       53     3.93e-05       2.41e-06      
auto       128    1.95e-03       1.25e-03      
auto       256    3.97e-03       4.17e-03      
auto       512    2.09e-02       1.49e-02      
leaver     53     6.78e-04       4.13e-07      
leaver     128    1.33e-01       2.25e-05      
leaver     256    2.39e-01       2.85e-05      
leaver     512    3.97e-01       5.17e-05      
chebyshev  53     2.35e-03       2.96e-07      
chebyshev  128    2.88e-01       2.93e-05      
chebyshev  256    3.19e-01       5.09e-05      
chebyshev  512    4.28e-01       1.21e-04      


## 12. Three points about the input

* `BigFloat(0.35)` is the binary value of the Float64 number `0.35`, not ``7/20``. Pass a rational with
  `precision`, or `parse(BigFloat, "0.35")`.
* `big"0.35"` is parsed when the code is compiled, at the precision in effect then; inside a function use
  `parse` or a rational.
* Whether two modes are distinguishable is judged in the precision of the input (section 5).

In [18]:
a = spin_weighted_spheroidal_eigenvalue(-2, 2, 2, BigFloat(0.35))    # c = 0.34999999999999997779…
b = spin_weighted_spheroidal_eigenvalue(-2, 2, 2, 7//20; precision=256)
Float64(abs(a - b))

1.4438281272670026e-16

See also the documentation page *Arbitrary precision* (`docs/src/precision.md`).